In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
ROOT = Path('..').resolve()
CACHE = ROOT / 'deep_data_audit' / 'cache'
OUT = ROOT / 'coursework_submission_notebooks' / 'outputs' / '02_fisher'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
Q_CUT = 0.05
FC4 = 4.0
FC32 = 32.0


In [ ]:
def load_sample(donor, tp):
    df = pd.read_parquet(CACHE / f'{donor}_{tp}_F1.parquet')
    cols = ['cdr3_nt', 'cdr3_aa', 'v_gene', 'd_gene', 'j_gene', 'duplicate_count']
    return df[cols].copy()

def fisher_one_donor(donor):
    d0 = load_sample(donor, '0')
    d15 = load_sample(donor, '15')
    N0 = int(d0['duplicate_count'].sum())
    N15 = int(d15['duplicate_count'].sum())
    a = d0.rename(columns={'duplicate_count': 'count_day0', 'cdr3_aa': 'cdr3_aa_0', 'v_gene': 'V_gene_0', 'd_gene': 'D_gene_0', 'j_gene': 'J_gene_0'})
    b = d15.rename(columns={'duplicate_count': 'count_day15', 'cdr3_aa': 'cdr3_aa_15', 'v_gene': 'V_gene_15', 'd_gene': 'D_gene_15', 'j_gene': 'J_gene_15'})
    m = a.merge(b, on='cdr3_nt', how='outer')
    m['count_day0'] = m['count_day0'].fillna(0).astype(np.int64)
    m['count_day15'] = m['count_day15'].fillna(0).astype(np.int64)
    m['cdr3_aa'] = m['cdr3_aa_15'].fillna(m['cdr3_aa_0'])
    m['V_gene'] = m['V_gene_15'].fillna(m['V_gene_0'])
    m['D_gene'] = m['D_gene_15'].fillna(m['D_gene_0'])
    m['J_gene'] = m['J_gene_15'].fillna(m['J_gene_0'])
    m['total_day0'] = N0
    m['total_day15'] = N15
    m['freq_day0'] = m['count_day0'] / N0
    m['freq_day15'] = m['count_day15'] / N15
    both = (m['count_day0'] > 0) & (m['count_day15'] > 0)
    only15 = (m['count_day0'] == 0) & (m['count_day15'] > 0)
    only0 = (m['count_day0'] > 0) & (m['count_day15'] == 0)
    m['detection_category'] = np.select([both, only15, only0], ['both', 'Day15_only', 'Day0_only'], default='unknown')
    fc = np.full(len(m), np.nan, dtype=float)
    fc[both.to_numpy()] = m.loc[both, 'freq_day15'].to_numpy() / m.loc[both, 'freq_day0'].to_numpy()
    fc[only15.to_numpy()] = np.inf
    fc[only0.to_numpy()] = 0.0
    m['fold_change'] = fc
    with np.errstate(divide='ignore', invalid='ignore'):
        m['log2_fold_change'] = np.log2(fc)
    n0 = m['count_day0'].to_numpy(dtype=np.int64)
    n15 = m['count_day15'].to_numpy(dtype=np.int64)
    M = N0 + N15
    p = hypergeom.sf(n15 - 1, M, n0 + n15, N15)
    m['p_value'] = p
    (_, q, _, _) = multipletests(p, method='fdr_bh')
    m['q_value'] = q
    m['fisher_positive_FC4'] = (m['q_value'] < Q_CUT) & (m['fold_change'] > FC4)
    m['fisher_positive_FC32'] = (m['q_value'] < Q_CUT) & (m['fold_change'] > FC32)
    m['donor'] = donor
    out = m[['donor', 'cdr3_nt', 'cdr3_aa', 'V_gene', 'D_gene', 'J_gene', 'count_day0', 'count_day15', 'total_day0', 'total_day15', 'freq_day0', 'freq_day15', 'fold_change', 'log2_fold_change', 'p_value', 'q_value', 'detection_category', 'fisher_positive_FC4', 'fisher_positive_FC32']].sort_values(['fisher_positive_FC32', 'fisher_positive_FC4', 'q_value'], ascending=[False, False, True])
    summary = {'donor': donor, 'N0': N0, 'N15': N15, 'n_tested': int(len(out)), 'n_positive_FC4': int(out['fisher_positive_FC4'].sum()), 'n_positive_FC32': int(out['fisher_positive_FC32'].sum()), 'FC4_both': int((out['fisher_positive_FC4'] & (out['detection_category'] == 'both')).sum()), 'FC4_Day15_only': int((out['fisher_positive_FC4'] & (out['detection_category'] == 'Day15_only')).sum())}
    return (out, summary)


In [ ]:
summaries = []
parts = []
for donor in DONORS:
    (df, summ) = fisher_one_donor(donor)
    print(donor, summ)
    df.to_parquet(TAB / f'fisher_all_clonotypes_{donor}.parquet', index=False)
    df[df.fisher_positive_FC4].to_csv(TAB / f'fisher_positive_FC4_{donor}.csv', index=False)
    summaries.append(summ)
    parts.append(df)
summary = pd.DataFrame(summaries)
summary.to_csv(TAB / 'summary_by_donor.csv', index=False)
all_pos = pd.concat([p[p.fisher_positive_FC4] for p in parts], ignore_index=True)
all_pos.to_csv(TAB / 'fisher_positive_FC4_all_donors.csv', index=False)
summary


In [ ]:
(fig, ax) = plt.subplots(figsize=(8, 4))
x = np.arange(len(summary))
ax.bar(x, summary['n_positive_FC4'], color='#C0392B')
ax.set_xticks(x)
ax.set_xticklabels(summary['donor'])

plt.tight_layout()
plt.savefig(FIG / 'n_positive_FC4.png', dpi=150, bbox_inches='tight')
plt.show()
(fig, axes) = plt.subplots(2, 3, figsize=(12, 7))
for (ax, donor, df) in zip(axes.ravel(), DONORS, parts):
    fin = np.isfinite(df['log2_fold_change'].to_numpy())
    dd = df.loc[fin]
    if len(dd) > 40000:
        dd = dd.sample(40000, random_state=0)
    y = -np.log10(dd['p_value'].clip(lower=1e-300))
    ax.scatter(dd['log2_fold_change'], y, s=3, c='0.7', alpha=0.2)
    sig = dd['q_value'] < Q_CUT
    ax.scatter(dd.loc[sig, 'log2_fold_change'], y[sig], s=5, c='#d62728', alpha=0.45)
    ax.axvline(np.log2(FC4), color='C0', ls='--', lw=1)

plt.tight_layout()
plt.savefig(FIG / 'volcano_by_donor.png', dpi=140, bbox_inches='tight')
plt.show()
